# 01 — EHR Data Understanding

## EHR Patient Journey & Clinical Outcomes Analytics

This notebook explores the structure, relationships, and quality of a synthetic longitudinal EHR dataset generated with Synthea.

### Objectives

- Understand the structure of the EHR-style dataset
- Identify relationships between patients, encounters, diagnoses, observations, medications, and procedures
- Evaluate data completeness and quality
- Understand the longitudinal patient journey
- Identify the clinical and operational variables available for cohort construction and outcomes analysis

### Core EHR Entities

| Dataset | Clinical Concept |
|---|---|
| patients | Patient demographics |
| encounters | Healthcare encounters |
| conditions | Diagnoses / clinical conditions |
| observations | Laboratory results, vital signs, and clinical observations |
| medications | Medication history |
| procedures | Clinical procedures |

The goal is not simply to analyze individual CSV files, but to understand how these entities interact to represent a longitudinal patient record.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

DATA_PATH = Path("../data/raw")

print("Data directory:", DATA_PATH.resolve())
print("\nAvailable files:")

for file in sorted(DATA_PATH.glob("*.csv")):
    print("-", file.name)

Data directory: C:\Projects\ehr-patient-journey-analytics\data\raw

Available files:
- allergies.csv
- careplans.csv
- claims.csv
- claims_transactions.csv
- conditions.csv
- devices.csv
- encounters.csv
- imaging_studies.csv
- immunizations.csv
- medications.csv
- observations.csv
- organizations.csv
- patients.csv
- payer_transitions.csv
- payers.csv
- procedures.csv
- providers.csv
- supplies.csv


## 2. Load Core EHR Tables

The analysis focuses initially on six core datasets representing the longitudinal patient journey:

- **Patients:** demographic and patient-level information
- **Encounters:** interactions with the healthcare system
- **Conditions:** diagnoses recorded during care
- **Observations:** laboratory results, vital signs, and other clinical measurements
- **Medications:** medication records
- **Procedures:** clinical procedures performed

These tables will later be linked using patient and encounter identifiers to reconstruct longitudinal clinical histories.

In [2]:
patients = pd.read_csv(DATA_PATH / "patients.csv")
encounters = pd.read_csv(DATA_PATH / "encounters.csv")
conditions = pd.read_csv(DATA_PATH / "conditions.csv")
observations = pd.read_csv(DATA_PATH / "observations.csv")
medications = pd.read_csv(DATA_PATH / "medications.csv")
procedures = pd.read_csv(DATA_PATH / "procedures.csv")

core_tables = {
    "patients": patients,
    "encounters": encounters,
    "conditions": conditions,
    "observations": observations,
    "medications": medications,
    "procedures": procedures
}

print("Core EHR tables loaded successfully.\n")

for name, df in core_tables.items():
    print(
        f"{name:<15} "
        f"rows: {df.shape[0]:>8,} | "
        f"columns: {df.shape[1]:>2}"
    )

Core EHR tables loaded successfully.

patients        rows:      108 | columns: 28
encounters      rows:    5,571 | columns: 15
conditions      rows:    3,517 | columns:  7
observations    rows:   68,648 | columns:  9
medications     rows:    3,850 | columns: 13
procedures      rows:   15,884 | columns: 10


## 3. EHR Table Structure and Key Fields

Before linking the datasets, the structure of each table must be examined to identify:

- Patient identifiers
- Encounter identifiers
- Clinical event timestamps
- Diagnosis, observation, medication, and procedure fields
- Potential primary and foreign keys

Understanding these relationships is essential for reconstructing a longitudinal patient journey without introducing duplicate records or incorrect joins.

In [3]:
for name, df in core_tables.items():
    print("=" * 80)
    print(name.upper())
    print("=" * 80)
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    
    for column in df.columns:
        print(f"  - {column}")
    
    print()

PATIENTS
Shape: (108, 28)

Columns:
  - Id
  - BIRTHDATE
  - DEATHDATE
  - SSN
  - DRIVERS
  - PASSPORT
  - PREFIX
  - FIRST
  - MIDDLE
  - LAST
  - SUFFIX
  - MAIDEN
  - MARITAL
  - RACE
  - ETHNICITY
  - GENDER
  - BIRTHPLACE
  - ADDRESS
  - CITY
  - STATE
  - COUNTY
  - FIPS
  - ZIP
  - LAT
  - LON
  - HEALTHCARE_EXPENSES
  - HEALTHCARE_COVERAGE
  - INCOME

ENCOUNTERS
Shape: (5571, 15)

Columns:
  - Id
  - START
  - STOP
  - PATIENT
  - ORGANIZATION
  - PROVIDER
  - PAYER
  - ENCOUNTERCLASS
  - CODE
  - DESCRIPTION
  - BASE_ENCOUNTER_COST
  - TOTAL_CLAIM_COST
  - PAYER_COVERAGE
  - REASONCODE
  - REASONDESCRIPTION

CONDITIONS
Shape: (3517, 7)

Columns:
  - START
  - STOP
  - PATIENT
  - ENCOUNTER
  - SYSTEM
  - CODE
  - DESCRIPTION

OBSERVATIONS
Shape: (68648, 9)

Columns:
  - DATE
  - PATIENT
  - ENCOUNTER
  - CATEGORY
  - CODE
  - DESCRIPTION
  - VALUE
  - UNITS
  - TYPE

MEDICATIONS
Shape: (3850, 13)

Columns:
  - START
  - STOP
  - PATIENT
  - PAYER
  - ENCOUNTER
  - CODE
  - DE

## 4. Identify Key EHR Relationship Fields

The EHR tables are linked through patient and encounter identifiers.

This step identifies the key fields needed to reconstruct the longitudinal patient journey across encounters, diagnoses, observations, medications, and procedures.

In [4]:
key_terms = [
    "ID", "PATIENT", "ENCOUNTER",
    "START", "STOP", "DATE",
    "CODE", "DESCRIPTION",
    "VALUE", "UNITS", "TYPE"
]

for name, df in core_tables.items():
    relevant_columns = [
        col for col in df.columns
        if any(term in col.upper() for term in key_terms)
    ]

    print(f"\n{name.upper()}")
    print("-" * 60)
    print(relevant_columns)


PATIENTS
------------------------------------------------------------
['Id', 'BIRTHDATE', 'DEATHDATE', 'MIDDLE', 'MAIDEN']

ENCOUNTERS
------------------------------------------------------------
['Id', 'START', 'STOP', 'PATIENT', 'PROVIDER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'REASONCODE', 'REASONDESCRIPTION']

CONDITIONS
------------------------------------------------------------
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION']

OBSERVATIONS
------------------------------------------------------------
['DATE', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'VALUE', 'UNITS', 'TYPE']

MEDICATIONS
------------------------------------------------------------
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'REASONCODE', 'REASONDESCRIPTION']

PROCEDURES
------------------------------------------------------------
['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'REASONCODE', 'REASONDESCRIPTION']


## 5. Encounter Structure

Encounters form the backbone of the longitudinal patient journey.

Each encounter connects a patient to a specific episode of care and can subsequently be linked to diagnoses, observations, medications, and procedures.

In [5]:
encounter_cols = [
    "Id",
    "START",
    "STOP",
    "PATIENT",
    "ENCOUNTERCLASS",
    "CODE",
    "DESCRIPTION",
    "BASE_ENCOUNTER_COST"
]

encounters[encounter_cols].head(10)

,Id,START,STOP,PATIENT,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST
0,cc3eac4a-34a7-9a15-13f3-57f4a6b934f9,1996-03-11T08:35:36Z,1996-03-11T09:21:33Z,cc3eac4a-34a7-9a15-1522-8087135631b7,wellness,162673000,General examination of patient (procedure),136.80
1,cc3eac4a-34a7-9a15-1dc5-a353457420c6,2006-03-27T08:35:36Z,2006-03-27T09:29:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,wellness,162673000,General examination of patient (procedure),136.80
2,cc3eac4a-34a7-9a15-94b5-26ac8c6c7134,2009-03-30T08:35:36Z,2009-03-30T09:25:14Z,cc3eac4a-34a7-9a15-1522-8087135631b7,wellness,162673000,General examination of patient (procedure),136.80
3,cc3eac4a-34a7-9a15-2a3d-1a1ba491a8ac,2012-04-02T08:35:36Z,2012-04-02T09:18:03Z,cc3eac4a-34a7-9a15-1522-8087135631b7,wellness,162673000,General examination of patient (procedure),136.80
4,cc3eac4a-34a7-9a15-99df-e3a0393fa95e,2015-04-06T08:35:36Z,2015-04-06T09:07:26Z,cc3eac4a-34a7-9a15-1522-8087135631b7,wellness,162673000,General examination of patient (procedure),136.80
5,cc3eac4a-34a7-9a15-d602-66b0bd935203,2016-03-28T04:35:36Z,2016-03-28T04:52:09Z,cc3eac4a-34a7-9a15-1522-8087135631b7,ambulatory,185345009,Encounter for symptom (procedure),85.55
6,cc3eac4a-34a7-9a15-25f1-0d3ead2180b6,2016-10-25T09:35:36Z,2016-10-25T09:50:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,ambulatory,185345009,Encounter for symptom (procedure),85.55
7,cc3eac4a-34a7-9a15-da83-de2c625ae978,2018-01-22T08:35:36Z,2018-01-22T09:34:09Z,cc3eac4a-34a7-9a15-1522-8087135631b7,wellness,162673000,General examination of patient (procedure),136.80
8,cc3eac4a-34a7-9a15-197d-c8faa82aee51,2018-02-05T08:35:36Z,2018-02-05T12:53:48Z,cc3eac4a-34a7-9a15-1522-8087135631b7,ambulatory,185349003,Encounter for check up (procedure),85.55
9,cc3eac4a-34a7-9a15-747e-7510e2463417,2020-01-27T08:35:36Z,2020-01-27T09:07:58Z,cc3eac4a-34a7-9a15-1522-8087135631b7,wellness,162673000,General examination of patient (procedure),136.80


### Encounter Types

Understanding the distribution of encounter classes provides context on how patients interact with the healthcare system.

In [6]:
encounter_summary = (
    encounters["ENCOUNTERCLASS"]
    .value_counts()
    .rename_axis("encounter_class")
    .reset_index(name="encounters")
)

encounter_summary["percent"] = (
    encounter_summary["encounters"] / len(encounters) * 100
).round(1)

encounter_summary

,encounter_class,encounters,percent
0,ambulatory,2939,52.8
1,wellness,1262,22.7
2,outpatient,759,13.6
3,emergency,270,4.8
4,urgentcare,195,3.5
5,home,63,1.1
6,inpatient,58,1.0
7,snf,9,0.2
8,hospice,8,0.1
9,virtual,8,0.1


## 6. Relational Integrity Checks

Before reconstructing patient journeys, the relationships between EHR tables were validated to identify orphaned patient or encounter references.

In [7]:
patient_ids = set(patients["Id"])
encounter_ids = set(encounters["Id"])

print("RELATIONAL INTEGRITY CHECK")
print("-" * 60)

print(
    "Encounters with unknown patient:",
    (~encounters["PATIENT"].isin(patient_ids)).sum()
)

for name, df in {
    "conditions": conditions,
    "observations": observations,
    "medications": medications,
    "procedures": procedures
}.items():

    unknown_patients = (~df["PATIENT"].isin(patient_ids)).sum()

    unknown_encounters = (
        (~df["ENCOUNTER"].isin(encounter_ids)).sum()
        if "ENCOUNTER" in df.columns
        else "N/A"
    )

    print(
        f"{name:<15} "
        f"unknown patients: {unknown_patients:<5} | "
        f"unknown encounters: {unknown_encounters}"
    )

RELATIONAL INTEGRITY CHECK
------------------------------------------------------------
Encounters with unknown patient: 0
conditions      unknown patients: 0     | unknown encounters: 0
observations    unknown patients: 0     | unknown encounters: 3060
medications     unknown patients: 0     | unknown encounters: 0
procedures      unknown patients: 0     | unknown encounters: 0


## 7. Data Understanding Summary

The EHR dataset follows a longitudinal relational structure in which patients are linked to encounters and clinical events including diagnoses, observations, medications, and procedures.

### Key findings

- The dataset contains 108 patients and 5,571 encounters.
- Encounter activity spans multiple care settings, including ambulatory, wellness, outpatient, emergency, urgent care, inpatient, home, hospice, SNF, and virtual care.
- Ambulatory encounters represent the largest share of recorded encounters (52.8%), followed by wellness encounters (22.7%) and outpatient encounters (13.6%).
- Patient identifiers are consistent across the core clinical tables examined.
- Conditions, medications, and procedures show complete encounter linkage to the encounter table.
- A subset of observations (3,060 records) contains encounter identifiers that are not represented in the encounters table. These records will require explicit handling when constructing encounter-level analytical datasets.

This structure supports reconstruction of longitudinal patient journeys across care settings while preserving the distinction between patient-level and encounter-level clinical information.